# 08 - OpenClaw GRU Training

Train the real `src/gru` model from the DB-first OpenClaw tables.

This notebook:
- loads the active DB-first training build
- loads imported OpenClaw tool-call rows
- runs the real GNN forward pass on leaf nodes to produce vocab embeddings
- trains the real GRU on leaf sequences
- persists `gnn_params` and `gru_weights` back into `.vault-exec/vault.kv`


In [ ]:
import {
  getActiveTrainingBuildId,
  listActiveSessionSequences,
  listActiveToolLeafEdgesNext,
  listActiveToolLeafNodes,
} from "../src/training-data/rebuild.ts";
import { trainGruFromOpenClawData } from "../src/training-data/gru-training.ts";
import { openVaultStore } from "../src/db/index.ts";
import { OpenClawLocalStore } from "../src/ingest/local-store.ts";
import { deserializeWeights } from "../src/gru/weights.ts";

const VAULT_PATH = "../demo-vault";
const DB_PATH = `${VAULT_PATH}/.vault-exec/vault.kv`;
const MIN_CALLS = 3;
const MAX_EXAMPLES = 512;
const MAX_EPOCHS = 1;
const INCLUDE_SUBAGENTS = true;

const kv = await Deno.openKv(DB_PATH);
const buildId = await getActiveTrainingBuildId(kv);
if (!buildId) {
  throw new Error(`No active training build found in ${DB_PATH}. Run 'deno task cli init ${VAULT_PATH}' first.`);
}
const nodeRows = await listActiveToolLeafNodes(kv);
const edgeRows = await listActiveToolLeafEdgesNext(kv);
const sequenceRows = await listActiveSessionSequences(kv);
kv.close();

const localStore = await OpenClawLocalStore.open(DB_PATH);
const toolCallRows = await localStore.listToolCalls();
localStore.close();

console.log(`Active build: ${buildId}`);
console.log(`Leaf nodes: ${nodeRows.length}`);
console.log(`Leaf transitions: ${edgeRows.length}`);
console.log(`Session sequences: ${sequenceRows.length}`);
console.log(`Imported tool calls: ${toolCallRows.length}`);
console.log(`Training cap: ${MAX_EXAMPLES} example(s), ${MAX_EPOCHS} epoch(s)`);


In [ ]:
const store = await openVaultStore(DB_PATH);
let trainingResult;
let latestWeights;
let decoded;

try {
  trainingResult = await trainGruFromOpenClawData(store, {
    nodeRows,
    edgeRows,
    toolCalls: toolCallRows,
    minCalls: MIN_CALLS,
    includeSubagents: INCLUDE_SUBAGENTS,
    maxExamples: MAX_EXAMPLES,
    maxEpochs: MAX_EPOCHS,
  });

  latestWeights = await store.getLatestWeights();
  if (!latestWeights) {
    throw new Error("GRU weights were not persisted.");
  }

  decoded = await deserializeWeights(latestWeights.blob);
} finally {
  store.close();
}

console.log(`GNN params source: ${trainingResult.paramsSource}`);
console.log(`Training examples used: ${trainingResult.exampleCount}`);
console.log(`Vocabulary size: ${trainingResult.vocabSize}`);
console.log(`Last avg loss: ${trainingResult.avgLoss.toFixed(4)}`);
console.log(`Last accuracy: ${(trainingResult.accuracy * 100).toFixed(1)}%`);
console.log(`Persisted epoch: ${latestWeights.epoch}`);
console.log(`Persisted vocab size: ${latestWeights.vocabSize}`);
console.log(`Decoded vocab size: ${decoded.vocab.indexToName.length}`);


In [ ]:
const retainedSequences = sequenceRows.filter((row) => row.callCount >= MIN_CALLS)
  .filter((row) => INCLUDE_SUBAGENTS || row.sessionKind !== "subagent");

const sessionKindCounts = new Map();
for (const row of retainedSequences) {
  sessionKindCounts.set(row.sessionKind, (sessionKindCounts.get(row.sessionKind) ?? 0) + 1);
}

const lengthRows = retainedSequences.map((row) => ({
  sessionId: row.sessionId,
  sessionKind: row.sessionKind,
  callCount: row.callCount,
})).sort((left, right) => right.callCount - left.callCount);

console.log(`Retained sequences: ${retainedSequences.length}`);
console.log(`Session-kind split:`);
console.table([...sessionKindCounts.entries()].map(([sessionKind, count]) => ({ sessionKind, count })));
console.log(`Top vocab tokens:`);
console.table(decoded.vocab.indexToName.slice(0, 20).map((leafKey, index) => ({
  index,
  leafKey,
})));


In [ ]:
const historySpec = {
  "$schema": "https://vega.github.io/schema/vega-lite/v5.json",
  title: "GRU training metrics (DB-first OpenClaw)",
  width: 720,
  height: 240,
  data: {
    values: trainingResult.history.flatMap((row) => ([
      { epoch: row.epoch, metric: "avgLoss", value: row.avgLoss },
      { epoch: row.epoch, metric: "accuracy", value: row.accuracy },
    ])),
  },
  mark: { type: "line", point: true },
  encoding: {
    x: { field: "epoch", type: "ordinal", title: "Epoch" },
    y: { field: "value", type: "quantitative", title: "Metric value" },
    color: { field: "metric", type: "nominal", title: "Metric" },
    tooltip: [
      { field: "epoch" },
      { field: "metric" },
      { field: "value", format: ".4f" },
    ],
  },
};

const lengthSpec = {
  "$schema": "https://vega.github.io/schema/vega-lite/v5.json",
  title: `Sequence length distribution (min_calls >= ${MIN_CALLS})`,
  width: 720,
  height: 260,
  data: { values: lengthRows },
  mark: "bar",
  encoding: {
    x: { field: "callCount", type: "quantitative", bin: true, title: "Calls per sequence" },
    y: { aggregate: "count", type: "quantitative", title: "Sequences" },
    color: { field: "sessionKind", type: "nominal", title: "Session kind" },
  },
};

historySpec;
lengthSpec;
